# DATA IMPORT

In [1]:
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(
    accept='.mp3,.wav',  
    multiple=False      
)
display(uploader)

FileUpload(value=(), accept='.mp3,.wav', description='Upload')

In [13]:
import base64

audio_file = uploader.value[0]
print(audio_file)

audio_bytes = audio_file['content']

# Se è un memoryview (comune in ipywidgets), convertilo in bytes
if isinstance(audio_bytes, memoryview):
    audio_bytes = audio_bytes.tobytes()
    
# # 2. Codifica in Base64
# audio_b64 = base64.b64encode(audio_bytes).decode('utf-8')

# 3. Identifica il tipo MIME (es. 'audio/mpeg' per mp3 o 'audio/wav')
mime_type = audio_file.get('type', 'audio/wav')

{'name': 'luvvoice.com-20260316_short.mp3', 'type': 'audio/mpeg', 'size': 56880, 'content': <memory at 0x000001FC63C7D780>, 'last_modified': datetime.datetime(2026, 3, 16, 15, 42, 58, 20000, tzinfo=datetime.timezone.utc)}


# HUGGINGFACE PIPELINE

In [20]:
import torch
from transformers import pipeline

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device="cuda" if torch.cuda.is_available() else "cpu", 
)

res_pipeline = asr_pipe(
    audio_bytes, 
    return_timestamps=True, # because the length of the data is major than 30 sec
)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

In [22]:
print(res_pipeline["chunks"])

[{'timestamp': (0.0, 5.02), 'text': ' This tool can generate a variety of character voices that you can use in marketing and social'}, {'timestamp': (5.02, 10.88), 'text': ' media, you can use to learn new languages and read books aloud. Copy and paste an existing'}, {'timestamp': (10.88, 16.5), 'text': ' script or type in the text for your script on text editor. Choose an AI voice of your choice'}, {'timestamp': (16.5, 23.56), 'text': ' from the library of voices. Some users said, this is a very good text reader and TTS tool.'}, {'timestamp': (24.2, 24.54), 'text': ''}, {'timestamp': (26.52, 27.56), 'text': ' It generates realistic AI voice.'}, {'timestamp': (29.28, None), 'text': " Believe me, you won't regret it."}]


In [23]:
print(res_pipeline["text"])

 This tool can generate a variety of character voices that you can use in marketing and social media, you can use to learn new languages and read books aloud. Copy and paste an existing script or type in the text for your script on text editor. Choose an AI voice of your choice from the library of voices. Some users said, this is a very good text reader and TTS tool. It generates realistic AI voice. Believe me, you won't regret it.


# HUGGINGFACE PIPELINE CHUNKED

In [26]:
import torch
from transformers import pipeline

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device="cuda" if torch.cuda.is_available() else "cpu", 
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
    chunk_length_s=3,
)

res_pipeline = asr_pipe(audio_bytes)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [27]:
print(res_pipeline) # compare this output with no-chunked version

{'text': " This tool can generate a variety of character voices that you can use in marketing and social media you can use to learn new languages and read books aloud. Copy and paste an existing script or type in the text for your script on the library of voices. Some users said. This is a very good text reader and TTS tool. It generates realistic AI voice. Believe me, you won't regret it."}


# LANGCHAIN AGENT BASED SOLUTION

In [14]:
import torch
from transformers import pipeline
from langchain.tools import tool

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device="cuda" if torch.cuda.is_available() else "cpu", 
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
)

@tool
def asr_converter(audio_b64: str) -> str:
    """Perform automatic speech recognition on the currently uploaded audio file."""
    audio_bytes = base64.b64decode(audio_b64)
    
    res = asr_pipe(audio_bytes, return_timestamps=True)
    
    return res["text"]

# @tool
# def asr_converter(dummy_input: str = "") -> str:
#     """Perform automatic speech recognition on the currently uploaded audio file."""
#     # Accede alla variabile audio_bytes definita globalmente nel notebook
#     global audio_bytes 
    
#     if not audio_bytes:
#         return "Error: No audio file correctly uploaded."
    
#     res = asr_pipe(audio_bytes, return_timestamps=True)
    
#     return res["text"]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="HuggingFaceTB/SmolLM2-360M-Instruct",
    task="text-generation",
)
model = ChatHuggingFace(llm=llm)

agent = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    tools = [asr_converter],
    system_prompt="You are a professional speech to text converter. Use your tools to perform Automatic Speech Recognition.",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [22]:
from langchain.messages import HumanMessage
audio_b64 = base64.b64encode(audio_bytes).decode('utf-8')


# CASE 1
# question = {
#     "role": "user",
#     "content": f"Convert this audio source into text:\n{audio_b64}"
# }
# config = {"configurable": {"thread_id": "1"}}

# response = agent.invoke({"messages": [question]}, config)


# CASE 2
question = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Convert this audio."},
        {"type": "audio", "base64": audio_b64, "mime_type": mime_type}
    ],
}
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [question]}, config)


# CASE 3
# question = [{"role": "user", "content": {"type": "text", "text": "Convert this audio."} },
#         {"role": "user", "content": {"type": "audio", "base64": audio_b64, "mime_type": mime_type}}, ]
   
# config = {"configurable": {"thread_id": "1"}}

# response = agent.invoke({"messages": question}, config)


# CASE 4
# multimodal_question = HumanMessage(content=[
#     {"type": "text", "text": "Convert this audio source into text."},
#     {"type": "audio", "base64": audio_b64, "mime_type": mime_type}
# ])
# config = {"configurable": {"thread_id": "1"}}

# response = agent.invoke({"messages": [multimodal_question]}, config)

TypeError: can only concatenate str (not "list") to str

In [5]:
# question = [{"role": "user", "content": "Convert the audio source provided in your tool."}]
# config = {"configurable": {"thread_id": "1"}}

# response = agent.invoke({"messages": question}, config)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

In [7]:
print(response["messages"][-1].content)
#print(response["messages"])

I’m ready to transcribe the audio for you. Could you please provide the audio file (or a link to it) so I can run the speech‑to‑text conversion? Once I have the file, I’ll process it and return the transcript.


# NEW FLOW

In [27]:
import torch
from transformers import pipeline
from langchain.tools import tool

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device="cuda" if torch.cuda.is_available() else "cpu", 
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
)

@tool
def asr_converter(dummy_input: str = "") -> str:
    """ ESSENTIAL tool for transcription.
    It MUST be used when the user indicates that an audio file is ready ([AUDIO_LOADED]).
    It automatically accesses the loaded audio buffer."""
    global audio_bytes
    if audio_bytes is None:
        return "Error: no audio in the buffer."
    res = asr_pipe(audio_bytes, return_timestamps=True)
    return res["text"]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [39]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_ollama import ChatOllama

# llm = HuggingFacePipeline.from_model_id(
#     model_id="HuggingFaceTB/SmolLM2-360M-Instruct",
#     task="text-generation",
# )
# model = ChatHuggingFace(llm=llm)

model = ChatOllama(model="gpt-oss:20b", temperature=0.1)
# model = ChatOllama(model="deepseek-r1:8b", temperature=0.1)

system_prompt = (
    "You are a professional ASR assistant. Your sole task is to transcribe audio."
    "If you see the tag [AUDIO_LOADED], you MUST immediately use the asr_converter tool"
    "without asking any questions. Do not describe what you are doing, just use the tool."
)
agent = create_agent(
    model = model,
    tools = [asr_converter],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

In [40]:
question = "Convert the loaded audio: [AUDIO_LOADED]"
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)

KeyError: 'num_frames'

In [26]:
print(response["messages"][-1].content)

<|im_start|>system
You are a professional ASR assistant. Your sole task is to transcribe audio.If you see the tag [AUDIO_LOADED], you MUST immediately use the asr_converter toolwithout asking any questions. Do not describe what you are doing, just use the tool.<|im_end|>
<|im_start|>user
Convert the loaded audio: [AUDIO_LOADED]<|im_end|>
<|im_start|>assistant
Loaded audio: [AUDIO_LOADED]
